In  a docker terminal run
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

then open jupyter in the printed link (such as http://127.0.0.1:8888/tree?token=4d9d578397aede50fc5bd2d92794b562a6bbcbc73b2dfe51)

CODE HERE, REFRESH AND EXECUTE IN THE BROWSER

In [1]:
import os
import sys
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"


In [2]:
sys.path.append('/app')

os.environ['DJANGO_SETTINGS_MODULE'] = 'app.settings'
os.environ['PYTHONPATH'] = '/app'

django.setup()

In [3]:
import numpy as np
import rasterio
from datetime import datetime, timedelta
from app import settings
from monica import models as models
from buek import models as buek_models
from django.db.models import Q, Min, Max
import os

In [4]:
germany_model_settings = models.GermanyModelParameters.objects.get(is_default=True)
# TODO: check with Claas or Marlene what parameters to use!  
cpp = germany_model_settings.to_json()

# load all relevant soil data for Germany- otherwise the query on each pixel would take too long
agri_buek = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'buek_id_agriculture_masked_4326.tif'))
altitude = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'dgm200_4326_1000m.tif'))
altitude_arr = altitude.read(1)
slope = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'dgm_1000_4326_slope.tif'))
slope_arr = slope.read(1)

# fetch all relevant soil profiles into memory
unique_buek_ids = np.unique(agri_buek.read(1))
unique_buek_ids = unique_buek_ids[unique_buek_ids != -9999]
soil_profiles = buek_models.SoilProfile.objects.filter(id__in=unique_buek_ids)
soil_profile_dict = {sp.id: sp.get_monica_horizons_json()[0] for sp in soil_profiles}
print("Soil profiles loaded")

Soil profiles loaded


In [5]:
def doy_to_iso(doy):
    if doy is None:
        return None
    date = datetime(2001, 1, 1) + timedelta(days=doy - 1)  # non-leap year
        # TODO: check if the first rotation is always 0000
    return f"0000-{date.strftime('%m-%d')}"

In [6]:
cultivar = germany_model_settings.cultivar

min_sowing_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Min('avg_sowing_doy'))['avg_sowing_doy__min']
max_harvest_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Max('avg_harvest_doy'))['avg_harvest_doy__max']
sowing_dates_list = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).values('climate_station__id', 'avg_sowing_doy', 'avg_harvest_doy')
sowing_dates_per_station = {data['climate_station__id']: {'sowing_date': doy_to_iso(data['avg_sowing_doy']), 'harvest_date': doy_to_iso(data['avg_harvest_doy'])} for data in sowing_dates_list}
print('sowing dates loaded', sowing_dates_per_station)
climate_stations_tif = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'nearest_station_per_cultivar', f'nearest_station_cultivar_{germany_model_settings.cultivar_name_for_sowing_dates}.tif'))
climate_stations_arr = climate_stations_tif.read(1)

sowing dates loaded {3: {'sowing_date': '0000-10-05', 'harvest_date': '0000-08-03'}, 98: {'sowing_date': '0000-09-29', 'harvest_date': '0000-08-12'}, 140: {'sowing_date': '0000-09-28', 'harvest_date': '0000-07-30'}, 164: {'sowing_date': '0000-09-21', 'harvest_date': '0000-07-20'}, 183: {'sowing_date': '0000-09-27', 'harvest_date': '0000-08-11'}, 198: {'sowing_date': '0000-10-10', 'harvest_date': '0000-07-26'}, 222: {'sowing_date': '0000-10-03', 'harvest_date': '0000-08-11'}, 282: {'sowing_date': '0000-10-07', 'harvest_date': '0000-07-30'}, 501: {'sowing_date': '0000-10-04', 'harvest_date': '0000-08-16'}, 524: {'sowing_date': '0000-10-15', 'harvest_date': '0000-08-08'}, 591: {'sowing_date': '0000-09-26', 'harvest_date': '0000-08-09'}, 596: {'sowing_date': '0000-09-25', 'harvest_date': '0000-08-06'}, 640: {'sowing_date': '0000-10-02', 'harvest_date': '0000-08-15'}, 662: {'sowing_date': '0000-10-07', 'harvest_date': '0000-08-02'}, 717: {'sowing_date': '0000-10-07', 'harvest_date': '0000-0

In [7]:
workstep = {
    "date":  '',               # "0000-10-13",
    "type": "Sowing",
    "crop": {
        # "is-winter-crop": True, # TODO is winter-crop is probably not required!!!
        "cropParams": {
            "species": {
            "=": cultivar.species_parameters.to_json()
            },
            "cultivar": {
            "=": cultivar.to_json()
            }
        },
        "residueParams": models.CropResidueParameters.objects.get(species_parameters=cultivar.species_parameters, is_default=True).to_json()
    }
}

In [8]:
agri_buek_as_array = agri_buek.read(1)
height, width = agri_buek_as_array.shape

In [9]:
start_date = '2025-08-01'
end_date = '2026-06-01'
lat_lon_idx_dictionary = models.DWDGridToPointIndices.get_lat_lon_dictionary()
climate_json = {
    "type": "DataAccessor",
    "data": None,
    "startDate": start_date,
    "endDate": end_date,
    }

In [10]:
events = [
    "daily",
        [
            "Date",
            "Yield",
            "LAI",
            "Stage",
            [
                "Mois",
            [
                1,
                20
            ]
            ],
            [
                "Mois",
            [
                1,
                10,
                "AVG"
            ]
            ],
            
        ]
    ]

In [13]:
with rasterio.open(
    os.path.join(settings.BASE_DIR, 
                 'monica', 'monica_geodata', '1000mx1000m', 'nearest_station_per_cultivar', 
                 f'nearest_station_cultivar_{germany_model_settings.cultivar_name_for_sowing_dates}.tif')) as f:
    climate_stations_tif = f.read(1)


In [14]:
climate_stations_tif

array([[-9999, -9999, -9999, ..., -9999, -9999, -9999],
       [-9999, -9999, -9999, ..., -9999, -9999, -9999],
       [-9999, -9999, -9999, ..., -9999, -9999, -9999],
       ...,
       [-9999, -9999, -9999, ..., -9999, -9999, -9999],
       [-9999, -9999, -9999, ..., -9999, -9999, -9999],
       [-9999, -9999, -9999, ..., -9999, -9999, -9999]], dtype=int32)

In [11]:
sowing_dates_per_station

{3: {'sowing_date': '0000-10-05', 'harvest_date': '0000-08-03'},
 98: {'sowing_date': '0000-09-29', 'harvest_date': '0000-08-12'},
 140: {'sowing_date': '0000-09-28', 'harvest_date': '0000-07-30'},
 164: {'sowing_date': '0000-09-21', 'harvest_date': '0000-07-20'},
 183: {'sowing_date': '0000-09-27', 'harvest_date': '0000-08-11'},
 198: {'sowing_date': '0000-10-10', 'harvest_date': '0000-07-26'},
 222: {'sowing_date': '0000-10-03', 'harvest_date': '0000-08-11'},
 282: {'sowing_date': '0000-10-07', 'harvest_date': '0000-07-30'},
 501: {'sowing_date': '0000-10-04', 'harvest_date': '0000-08-16'},
 524: {'sowing_date': '0000-10-15', 'harvest_date': '0000-08-08'},
 591: {'sowing_date': '0000-09-26', 'harvest_date': '0000-08-09'},
 596: {'sowing_date': '0000-09-25', 'harvest_date': '0000-08-06'},
 640: {'sowing_date': '0000-10-02', 'harvest_date': '0000-08-15'},
 662: {'sowing_date': '0000-10-07', 'harvest_date': '0000-08-02'},
 717: {'sowing_date': '0000-10-07', 'harvest_date': '0000-08-18'}

In [ ]:
for lat_idx in range(0, height):
        for lon_idx in range(0, width):
            if agri_buek_as_array[lat_idx][lon_idx] and agri_buek_as_array[lat_idx][lon_idx] != -9999:

                indices_dict = lat_lon_idx_dictionary[lat_idx][lon_idx]
                print(f"Running cell {lat_idx}, {lon_idx}")

                buek_id = int(agri_buek_as_array[lat_idx, lon_idx])
                soil_profile = soil_profile_dict.get(buek_id, None)